# Device & Sensor Telemetry Collection · Edge-Computing Notebook

Edge systems begin with **telemetry**: the stream of measurements a device and its sensors produce over time. Before you can store, visualize, or run AI on edge data, you have to **collect** it reliably.

In this lab you will build a telemetry collector on the DGX Spark. You will read the device's own real sensors, add a simulated environment sensor, and write structured readings to a local log file as **JSON lines**: one self-contained reading per line.

```text
Device sensors (real)  +  Environment sensor (simulated)
        -> Telemetry Collector
        -> JSON-lines log file on the edge node
```

The JSON-lines file you produce here is the input for the next lab (Grafana & Time-Series Database), where the same kind of readings get stored and visualized.

The main idea is telemetry is **structured measurement over time**. Good telemetry has a clear source, consistent units, and a timestamp on every reading.

## How this notebook works · cells and a Jupyter terminal

Most steps run once and return (writing a file, building an image) — those are ordinary notebook cells. The collector itself runs **forever** by design, and `docker logs -f` follows output live; a notebook cell cannot do those without hanging the kernel. For quick looks this notebook runs the collector under `timeout` (a few seconds, then it returns); for the real live experience it writes a small script and asks you to run it in a Jupyter terminal.

Two labels are used throughout:

- **[Notebook cell]** · run the cell right here.
- **[Terminal]** · open a Jupyter terminal — **File ▸ New ▸ Terminal** (or the blue **+** Launcher, then the **Terminal** tile) — and run the given command there. **Ctrl-C** stops a live run.

### One config, everywhere

Terminal shells do not share the notebook kernel's variables. So the setup cell below writes `~/telemetryLab/labEnv.sh` with your `DEVICE_ID`, and every helper script starts with `source ~/telemetryLab/labEnv.sh` — the notebook and your terminals always agree. (This lab needs no network ports.)

**Requirements:** run this notebook with a Python 3 kernel on the DGX Spark, where Docker is available to your user.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

---
## Part 0 · Before You Begin

You are working on a shared **DGX Spark** edge device, already connected with your own student account (set up in the *SSH, Linux, Git, GitHub* lab). **Every command in this exercise runs on the DGX Spark** — this notebook runs there through JupyterLab.

Because the DGX Spark is shared:

- use your own home folder and keep your files under it
- prefix any shared resource names with your account name using `${USER}`
- use only the ports assigned to you by the instructor (none are needed in this lab)

---
## Part 1 · Set Up Your Working Folder

Give your device a telemetry identity so its readings can be told apart from other students'. The original lab exports `DEVICE_ID=${USER}-thor` by hand in each shell; here the cell derives it from your username, exports it for notebook cells, creates the lab folder with a `data/` directory for collected telemetry, and writes `labEnv.sh` so Jupyter-terminal helper scripts agree.

**[Notebook cell]**

In [ ]:
# This lab needs no network ports - just your telemetry identity.
import os
import pathlib

if not os.environ.get("USER"):
    os.environ["USER"] = "student01"
userName = os.environ["USER"]
os.environ["DEVICE_ID"] = userName + "-thor"

labDir = pathlib.Path.home() / "telemetryLab"
(labDir / "data").mkdir(parents=True, exist_ok=True)
envText = (
    "# telemetryLab shared environment - sourced by every helper script.\n"
    'export DEVICE_ID="${USER}-thor"\n'
)
(labDir / "labEnv.sh").write_text(envText)
showEnvCard(envText, title="telemetryLab/labEnv.sh",
            envVars={"USER": userName, "DEVICE_ID": os.environ["DEVICE_ID"]})

In [ ]:
%cd ~/telemetryLab

### Preflight · check your environment

**[Notebook cell]** Run this once at the start. It confirms Python, Docker (needed in Part 6), and the Linux sensor interfaces the collector reads:

In [ ]:
import glob
import os
preflight([
    check("python3 available", commandOnPath("python3"),
          hint="The collector is a Python 3 script - python3 should ship with the "
               "DGX Spark's OS image; ask your instructor if it is missing.",
          link="https://docs.python.org/3/using/unix.html",
          linkText="Python on Unix"),
    check("docker daemon", dockerDaemonUp(),
          hint="Part 6 containerizes the collector - Docker must be running and your "
               "rootless Podman socket must be provisioned; ask your instructor.",
          link="https://docs.docker.com/engine/daemon/troubleshoot/",
          linkText="Docker daemon troubleshooting"),
    check("/proc/loadavg readable (CPU sensor)", fileExists("/proc/loadavg"),
          hint="/proc/loadavg exists on every Linux system - if this fails, you are "
               "probably not running the notebook on the DGX Spark.",
          link="https://man7.org/linux/man-pages/man5/proc.5.html",
          linkText="proc(5) man page"),
    check("/proc/meminfo readable (memory sensor)", fileExists("/proc/meminfo"),
          hint="/proc/meminfo exists on every Linux system - if this fails, you are "
               "probably not running the notebook on the DGX Spark.",
          link="https://man7.org/linux/man-pages/man5/proc.5.html",
          linkText="proc(5) man page"),
], infoRows=[("your USER", os.environ.get("USER", "?")),
             ("your DEVICE_ID", os.environ.get("DEVICE_ID", "?")),
             ("thermal zones found",
              len(glob.glob("/sys/class/thermal/thermal_zone*")))])

### Checkpoint · Part 1

In [ ]:
import os
checkpoint("Part 1 - workspace and identity", [
    check("lab data folder ready", dirExists("~/telemetryLab/data"),
          hint="Re-run the Part 1 setup cell - it creates ~/telemetryLab/data.",
          link="https://man7.org/linux/man-pages/man1/mkdir.1.html",
          linkText="mkdir man page"),
    check("labEnv.sh written", fileContains("~/telemetryLab/labEnv.sh", "DEVICE_ID"),
          hint="Re-run the Part 1 setup cell - terminal helper scripts source this file.",
          link="https://www.gnu.org/software/bash/manual/html_node/Environment.html",
          linkText="Bash environment variables"),
    check("DEVICE_ID exported to the kernel", envVarSet("DEVICE_ID"),
          hint="Re-run the Part 1 setup cell so notebook `!` cells see DEVICE_ID.",
          link="https://www.gnu.org/software/bash/manual/html_node/Environment.html",
          linkText="Bash environment variables"),
], successNote="Identity set: your readings will carry device_id '"
               + os.environ.get("DEVICE_ID", "?") + "'.",
   docLink="https://jsonlines.org/",
   docLinkText="JSON Lines format")

---
## Part 2 · What Telemetry Is

Telemetry is measurement that is collected automatically and tagged with time. Two common sources at the edge:

```text
Device sensors      -> the edge node measuring itself (CPU load, memory, temperature, power)
Environment sensors -> the world around the device (temperature, humidity, motion, a camera)
```

Every good telemetry reading answers three questions:

```text
When?   -> a timestamp
Where?  -> a device / sensor identity
What?   -> the measured values, with known units
```

The DGX Spark is itself a device full of real sensors. You will start by reading those.

---
## Part 3 · Read the DGX Spark's Built-In Sensors

The device exposes real sensor data through the Linux filesystem and system tools. No extra hardware is needed.

**[Notebook cell]** Read CPU load (the first three numbers are the 1, 5, and 15 minute load averages) and memory totals (in kB):

In [ ]:
!cat /proc/loadavg
!grep -E "MemTotal|MemAvailable" /proc/meminfo

**[Notebook cell]** Read the device temperatures from the thermal zones (values are in milli-degrees Celsius):

In [ ]:
%%bash
for zone in /sys/class/thermal/thermal_zone*/; do
  echo "$(cat "$zone/type"): $(cat "$zone/temp")"
done

> These are genuine sensor readings, but the DGX Spark is shared, so load and memory reflect **all** accounts' activity, not just yours. Temperature is a property of the whole board.

Everything above is a raw sensor value. Telemetry collection means sampling values like these on a schedule and recording them in a consistent structure.

### Checkpoint · Part 3

In [ ]:
checkpoint("Part 3 - built-in sensors readable", [
    check("CPU load sensor", fileExists("/proc/loadavg"),
          hint="/proc/loadavg should exist on any Linux system - confirm the notebook "
               "kernel runs on the DGX Spark, then re-run the Part 3 cells.",
          link="https://man7.org/linux/man-pages/man5/proc.5.html",
          linkText="proc(5) man page"),
    check("memory sensor", fileContains("/proc/meminfo", "MemTotal"),
          hint="/proc/meminfo should report MemTotal on any Linux system - confirm the "
               "notebook kernel runs on the DGX Spark.",
          link="https://man7.org/linux/man-pages/man5/proc.5.html",
          linkText="proc(5) man page"),
], successNote="The DGX Spark reports its own load and memory - real device sensors, no "
               "extra hardware. (Thermal zones are optional: a missing one records "
               "null later, not a failure.)",
   docLink="https://www.kernel.org/doc/Documentation/thermal/sysfs-api.txt",
   docLinkText="Linux thermal sysfs API")

---
## Part 4 · Define the Telemetry Reading Format

Before writing a collector, decide the shape of one reading. A clear, fixed structure is a **data contract**: every reading looks the same, so anything downstream can parse it.

You will use one JSON object per reading:

```json
{
  "timestamp": "2026-06-25T14:03:01Z",
  "device_id": "yourname-thor",
  "source": "device",
  "metrics": {
    "cpu_load_1m": 1.23,
    "mem_used_percent": 41.2,
    "temp_c": 47.5
  }
}
```

Fields:

```text
timestamp -> UTC time the reading was taken (ISO 8601)
device_id -> which device produced it
source    -> "device" or "environment"
metrics   -> the measured values, with known units in the key names
```

Writing one such object per line in a file is called **JSON Lines** (`.jsonl`). It is a common telemetry format because each line is a complete, independent record.

> Note the two naming worlds: the *data* keys (`device_id`, `cpu_load_1m`, `temp_c`) are part of the contract and stay exactly as defined here — the next lab's database expects them — while the *code* around them follows this course's camelCase Python style.

---
## Part 5 · Build the Telemetry Collector

Now build the collector. It reads the DGX Spark's **real** sensors (from Part 3) and adds a **simulated** environment sensor, emits both in the same reading format, and writes one JSON line per reading to a file — the telemetry the next lab will store. Both sources share the contract from Part 4; only `source` and `metrics` differ, which is exactly what lets one pipeline handle many sensor types.

**[Notebook cell]** Write the collector (the original lab uses `nano`; here the notebook writes it for you):

In [ ]:
%%writefile telemetryCollector.py
import os
import json
import time
import glob
import random
from datetime import datetime, timezone

deviceID = os.environ.get("DEVICE_ID", "thor")
interval = float(os.environ.get("INTERVAL", "2"))
logFile = os.environ.get("LOG_FILE", "data/telemetry.jsonl")


def nowUtc():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


# --- real device sensors: the DGX Spark measuring itself ---
def readCPULoad1m():
    with open("/proc/loadavg") as loadFile:
        return float(loadFile.read().split()[0])


def readMemUsedPercent():
    memInfo = {}
    with open("/proc/meminfo") as memFile:
        for line in memFile:
            key, value = line.split(":")
            memInfo[key] = float(value.strip().split()[0])
    total = memInfo.get("MemTotal", 0)
    available = memInfo.get("MemAvailable", 0)
    return round((total - available) / total * 100, 1) if total else 0.0


def readTempC():
    temps = []
    for path in glob.glob("/sys/class/thermal/thermal_zone*/temp"):
        try:
            with open(path) as tempFile:
                temps.append(int(tempFile.read().strip()) / 1000.0)
        except (ValueError, OSError):
            continue
    return round(max(temps), 1) if temps else None  # a failed sensor records null, not a crash


def deviceReading():
    return {
        "timestamp": nowUtc(),
        "device_id": deviceID,
        "source": "device",
        "metrics": {
            "cpu_load_1m": readCPULoad1m(),
            "mem_used_percent": readMemUsedPercent(),
            "temp_c": readTempC(),
        },
    }


# --- simulated environment sensor: the world around the device ---
# real sensors (I2C/USB/etc.) aren't wired to every DGX Spark, so we simulate and keep the same shape
def environmentReading():
    return {
        "timestamp": nowUtc(),
        "device_id": deviceID,
        "source": "environment",
        "metrics": {
            "temperature": round(random.uniform(22, 40), 2),
            "humidity": round(random.uniform(35, 60), 2),
            "motion": random.choice([True, False]),
            "battery": random.randint(20, 100),
        },
    }


if __name__ == "__main__":
    os.makedirs(os.path.dirname(logFile) or ".", exist_ok=True)
    print(f"Collecting telemetry for {deviceID} every {interval}s -> {logFile}", flush=True)

    with open(logFile, "a") as outFile:
        while True:
            for reading in (deviceReading(), environmentReading()):
                line = json.dumps(reading)
                print(line, flush=True)
                outFile.write(line + "\n")
                outFile.flush()
            time.sleep(interval)

In [ ]:
# Preview the collector we just wrote.
showFile("~/telemetryLab/telemetryCollector.py")

**[Notebook cell]** The collector loops forever, so the cell runs it under `timeout` for ~8 seconds and then stops it for you (`|| true` keeps the expected timeout exit quiet):

In [ ]:
!cd ~/telemetryLab && timeout 8 python3 telemetryCollector.py || true

**You should see:** two JSON lines every ~2 seconds: one `source: "device"` reading with live CPU/memory/temperature, and one `source: "environment"` reading with simulated values. If `temp_c` shows `null`, this DGX Spark doesn't expose a readable thermal zone to your account; the rest of the reading is still valid.

**[Notebook cell]** For an open-ended live run, write a helper script and run it in a terminal instead:

In [ ]:
%%writefile runCollector.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/telemetryLab/labEnv.sh
cd ~/telemetryLab

echo "Collecting telemetry for ${DEVICE_ID}. Press Ctrl-C to stop."
python3 telemetryCollector.py

In [ ]:
!chmod +x runCollector.sh

**[Terminal]** Watch it stream, then stop with **Ctrl-C**:

```bash
~/telemetryLab/runCollector.sh
```

**[Notebook cell]** Inspect what was collected:

In [ ]:
!wc -l ~/telemetryLab/data/telemetry.jsonl
!tail -n 4 ~/telemetryLab/data/telemetry.jsonl

You now have a growing telemetry log, two readings per cycle, every reading self-contained.

### Checkpoint · Part 5

This is the heart of the lab: the checks parse your actual log, line by line, against the Part 4 data contract.

In [ ]:
import os
dataFile = "~/telemetryLab/data/telemetry.jsonl"
checkpoint("Part 5 - collector produces contract-valid JSON lines", [
    check("telemetryCollector.py exists",
          fileExists("~/telemetryLab/telemetryCollector.py"),
          hint="Re-run the %%writefile telemetryCollector.py cell in Part 5.",
          link="https://docs.python.org/3/library/json.html",
          linkText="Python json module"),
    check("every line is valid JSON with the contract keys",
          jsonLinesValid(dataFile,
                         requiredKeys=["timestamp", "device_id", "source", "metrics"],
                         minRecords=4),
          hint="Each line must be one JSON object with timestamp, device_id, source, "
               "and metrics. Re-run the 8-second collector cell; if bad lines remain "
               "from an earlier attempt, delete data/telemetry.jsonl and collect again.",
          link="https://jsonlines.org/",
          linkText="JSON Lines format"),
    check("device readings present (source: \"device\")",
          fileContains(dataFile, '"source": "device"'),
          hint="No real-sensor readings found - re-run the collector cell and check "
               "its output for errors reading /proc.",
          link="https://man7.org/linux/man-pages/man5/proc.5.html",
          linkText="proc(5) man page"),
    check("environment readings present (source: \"environment\")",
          fileContains(dataFile, '"source": "environment"'),
          hint="No simulated environment readings found - the collector should emit "
               "one per cycle; re-run the %%writefile cell, then collect again.",
          link="https://jsonlines.org/",
          linkText="JSON Lines format"),
    check("readings tagged with your DEVICE_ID",
          fileContains(dataFile, os.environ.get("DEVICE_ID", "-thor")),
          hint="Readings carry the wrong identity - re-run the Part 1 setup cell "
               "(exports DEVICE_ID), then the collector cell.",
          link="https://www.gnu.org/software/bash/manual/html_node/Environment.html",
          linkText="Environment variables"),
], successNote="Structured, timestamped, contract-valid telemetry is flowing - "
               "exactly what the time-series database lab will ingest.",
   docLink="https://jsonlines.org/",
   docLinkText="JSON Lines format")

---
## Part 6 · Containerize the Collector

Edge collectors usually run as a container so they start automatically and stay isolated. Put the collector in a container that writes to a **Docker volume** for persistence.

**[Notebook cell]** Create the Dockerfile:

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY telemetryCollector.py .

ENV LOG_FILE=/data/telemetry.jsonl

CMD ["python", "telemetryCollector.py"]

In [ ]:
# Preview the Dockerfile we just wrote.
showFile("~/telemetryLab/Dockerfile")

**[Notebook cell]** Build the image and create a volume for the telemetry data:

> **Naming reminder:** the image tag stays lowercase (`${USER}-telemetry-collector`) while the container it runs is camelCase (`${USER}-telemetryCollector`). Docker requires image names to be lowercase; container names may use capitals. See Lab 02, Part 13 for the full explanation.

In [ ]:
!docker build -t ${USER}-telemetry-collector .
!docker volume inspect ${USER}-telemetryData >/dev/null 2>&1 || docker volume create ${USER}-telemetryData

**[Notebook cell]** Run the collector container. A container already sees the host's CPU load and memory through its own `/proc`, so those need no special setup; the thermal sensors under `/sys` are not present in a container by default, so mount them in read-only. Any previous run is removed first, so the cell is safe to re-run:

In [ ]:
!docker rm -f ${USER}-telemetryCollector 2>/dev/null || true
!docker run -d \
  --name ${USER}-telemetryCollector \
  -e DEVICE_ID=${USER}-thor \
  -v ${USER}-telemetryData:/data \
  -v /sys/class/thermal:/sys/class/thermal:ro \
  ${USER}-telemetry-collector

> The environment sensor is fully simulated, so it always works. If the thermal mount is not available on this DGX Spark, `temp_c` simply comes back as `null` while the rest of the reading still records. That is expected behavior for telemetry — a missing sensor should not stop the whole stream.

**[Notebook cell]** Confirm it is running and peek at recent output:

In [ ]:
import os
dockerPs(namePattern=os.environ.get("USER", "student01") + "-telemetryCollector")
dockerLogs(os.environ.get("USER", "student01") + "-telemetryCollector", tail=6)

**[Notebook cell]** `docker logs -f` follows the stream live and never returns, so it goes in a terminal. Write the watcher script:

In [ ]:
%%writefile watchCollector.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/telemetryLab/labEnv.sh

echo "Following live collector output. Ctrl-C stops watching; the container keeps running."
docker logs -f "${USER}-telemetryCollector"


In [ ]:
!chmod +x watchCollector.sh

**[Terminal]** Watch the live readings, then stop watching with **Ctrl-C** (the container keeps running):

```bash
~/telemetryLab/watchCollector.sh
```

**You should see:** the same JSON lines as before, now coming from inside the container, two per cycle (one `device` reading, one `environment` reading).

> **If it doesn't work:** if `docker ps` doesn't list `${USER}-telemetryCollector`, the container exited; run `docker logs ${USER}-telemetryCollector` (without `-f`) to see the error. A name-already-in-use error means a previous run is still there; remove it with `docker rm -f ${USER}-telemetryCollector` and re-run.

### Checkpoint · Part 6

In [ ]:
import os
userName = os.environ.get("USER", "student01")
checkpoint("Part 6 - collector containerized", [
    check("collector image built", imageExists(userName + "-telemetry-collector"),
          hint="Re-run the docker build cell in Part 6.",
          link="https://docs.docker.com/reference/cli/docker/buildx/build/",
          linkText="docker build reference"),
    check("telemetry volume exists", volumeExists(userName + "-telemetryData"),
          hint="Re-run the 'docker volume create' line in Part 6.",
          link="https://docs.docker.com/engine/storage/volumes/",
          linkText="Docker volumes"),
    check("collector container running",
          containerRunning(userName + "-telemetryCollector"),
          hint="Re-run the docker run cell in Part 6; if it exits immediately, read "
               "'docker logs " + userName + "-telemetryCollector' for the error.",
          link="https://docs.docker.com/reference/cli/docker/container/run/",
          linkText="docker run reference"),
    check("container is emitting readings",
          outputContains(["docker", "logs", "--tail", "5",
                          userName + "-telemetryCollector"], '"source":'),
          hint="The container runs but prints no readings - check its logs for a "
               "Python traceback, fix the %%writefile collector cell, rebuild, and "
               "re-run.",
          link="https://docs.docker.com/reference/cli/docker/container/logs/",
          linkText="docker logs reference"),
], successNote="The collector now runs as an isolated, restartable edge service with "
               "persistent storage.",
   docLink="https://docs.docker.com/engine/storage/volumes/",
   docLinkText="Docker volumes")

---
## Part 7 · Inspect the Collected Telemetry

The original lab opens an interactive shell inside the container (`docker exec -it ... sh`). A notebook cell has no TTY, so the same commands run non-interactively here — for the interactive version, run `docker exec -it ${USER}-telemetryCollector sh` in a **[Terminal]**.

**[Notebook cell]** Look at the telemetry stored in the volume from inside the container:

In [ ]:
!docker exec ${USER}-telemetryCollector wc -l /data/telemetry.jsonl
!docker exec ${USER}-telemetryCollector tail -n 6 /data/telemetry.jsonl

**[Notebook cell]** If `jq` is installed on the DGX Spark, pull just the device temperatures out of the stream:

In [ ]:
%%bash
command -v jq >/dev/null || { echo "jq not installed on this DGX Spark - use the grep cell below instead"; exit 0; }
docker exec "${USER}-telemetryCollector" cat /data/telemetry.jsonl \
  | jq -c 'select(.source == "device") | {timestamp, temp_c: .metrics.temp_c}' \
  | tail -n 5

**[Notebook cell]** Without `jq`, you can still filter the raw lines with `grep`:

In [ ]:
%%bash
docker exec "${USER}-telemetryCollector" cat /data/telemetry.jsonl \
  | grep '"source": "device"' \
  | tail -n 5

Because each line is independent JSON, telemetry is easy to filter, count, and forward. This is exactly why the next lab can take this stream straight into a time-series database.

### Checkpoint · Part 7

The last check re-parses the newest records **inside the volume** against the Part 4 contract — proof the containerized stream is as clean as the local one.

In [ ]:
import json
import os
userName = os.environ.get("USER", "student01")
containerName = userName + "-telemetryCollector"

def volumeRecordsValid():
    out, codeNum = runShell(["docker", "exec", containerName,
                             "tail", "-n", "5", "/data/telemetry.jsonl"])
    if codeNum != 0:
        return False, "could not read /data/telemetry.jsonl inside the container"
    goodCount = 0
    for line in out.strip().splitlines():
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            return False, "not valid JSON: " + line[:60]
        if all(key in record for key in ("timestamp", "device_id", "source", "metrics")):
            goodCount += 1
    return goodCount >= 3, str(goodCount) + " of the last 5 volume records match the contract"

checkpoint("Part 7 - telemetry inspected in the volume", [
    check("volume log is non-empty",
          commandSucceeds(["docker", "exec", containerName,
                           "test", "-s", "/data/telemetry.jsonl"]),
          hint="The container has not written anything yet - give it a few seconds "
               "after the Part 6 run cell, or check 'docker logs " + containerName + "'.",
          link="https://docs.docker.com/reference/cli/docker/container/exec/",
          linkText="docker exec reference"),
    check("newest volume records match the data contract", volumeRecordsValid,
          hint="Records inside the volume are malformed - rebuild the image after "
               "fixing the collector (%%writefile cell in Part 5), remove the volume "
               "with 'docker volume rm " + userName + "-telemetryData' after "
               "'docker rm -f " + containerName + "', and re-run Part 6.",
          link="https://jsonlines.org/",
          linkText="JSON Lines format"),
], successNote="The containerized stream is clean and queryable - filter it, count it, "
               "forward it.",
   docLink="https://jqlang.org/manual/",
   docLinkText="jq manual")

---
## Part 8 · Connecting Real Hardware Sensors

This lab simulates the environment sensor so it runs on any DGX Spark. Real edge sensors usually attach in one of these ways:

```text
I2C / SPI   -> small sensor chips (temperature, humidity, IMU)
GPIO        -> simple digital inputs (motion, buttons, switches)
USB         -> webcams, serial sensors, microphones
CSI         -> camera modules for computer vision
```

To use a real sensor, you would replace the simulated `environmentReading()` with code that reads the actual device (for example, a Python library for an I2C sensor, or OpenCV for a camera) and fill the same `metrics` structure. **Your instructor will provide any real sensor hardware and the library to read it.** Because the reading format is fixed, the rest of your pipeline does not change when you swap simulated data for real data.

---
## Part 9 · Telemetry Quality and Sampling Rate

Collecting data is easy; collecting **good** data takes design. Think about:

```text
Sampling rate -> faster sampling = more detail but more CPU, storage, and bandwidth
Units         -> keep them in the key name (temp_c, mem_used_percent) so they are never guessed
Timestamps    -> always UTC, always on every reading, so data can be ordered and compared
Missing data  -> a failed sensor should record null, not crash the collector
```

**[Notebook cell]** Try changing the sampling rate and watch the cost. Run the collector 4x faster for ~6 seconds, writing to a separate file:

In [ ]:
!cd ~/telemetryLab && timeout 6 env INTERVAL=0.5 LOG_FILE=data/fast.jsonl python3 telemetryCollector.py || true

**[Notebook cell]** Compare file growth:

In [ ]:
!wc -l ~/telemetryLab/data/telemetry.jsonl ~/telemetryLab/data/fast.jsonl

A higher sampling rate produces far more data. On a constrained, shared edge node, the right rate is a design decision: enough resolution to be useful, not so much that it overwhelms storage and the network. This is the same trade-off you weighed in the System Architecture lab.

### Checkpoint · Part 9

In [ ]:
import json
import pathlib

def fastSamplingDense():
    fastPath = pathlib.Path("~/telemetryLab/data/fast.jsonl").expanduser()
    if not fastPath.is_file():
        return False, "missing: " + str(fastPath)
    lines = [line for line in fastPath.read_text().splitlines() if line.strip()]
    if len(lines) < 8:
        return False, ("only " + str(len(lines)) + " record(s) - re-run the "
                       "INTERVAL=0.5 cell for its full 6 seconds")
    firstStamp = json.loads(lines[0])["timestamp"]
    lastStamp = json.loads(lines[-1])["timestamp"]
    return True, (str(len(lines)) + " records between " + firstStamp
                  + " and " + lastStamp)

checkpoint("Part 9 - sampling-rate experiment", [
    check("fast log is contract-valid too",
          jsonLinesValid("~/telemetryLab/data/fast.jsonl",
                         requiredKeys=["timestamp", "device_id", "source", "metrics"],
                         minRecords=8),
          hint="Re-run the INTERVAL=0.5 cell in Part 9 - it should produce at least "
               "8 records in 6 seconds (two per half-second cycle).",
          link="https://jsonlines.org/",
          linkText="JSON Lines format"),
    check("fast sampling produced a dense burst", fastSamplingDense,
          hint="Re-run the INTERVAL=0.5 cell and let it run its full 6 seconds.",
          link="https://docs.influxdata.com/influxdb/v2/write-data/best-practices/optimize-writes/",
          linkText="Write-rate trade-offs (InfluxDB)"),
    check("normal-rate log still intact",
          fileNonEmpty("~/telemetryLab/data/telemetry.jsonl", minLines=4),
          hint="The fast run wrote to data/fast.jsonl and must not disturb "
               "data/telemetry.jsonl - if it is gone, re-run the Part 5 collector cell.",
          link="https://jsonlines.org/",
          linkText="JSON Lines format"),
], successNote="You measured the cost of resolution: same collector, 4x the rate, "
               "several times the data.",
   docLink="https://docs.influxdata.com/influxdb/v2/write-data/best-practices/",
   docLinkText="Telemetry write best practices")

---
## Part 10 · Connect to the Rest of the Course

You built the first stage of the edge data pipeline:

```text
Sensors (device + environment)
    -> Telemetry Collector  (this lab)
    -> Time-Series Database + Grafana  (next lab)
    -> Dashboards, alerts, and AI on the stream  (later labs)
```

A real project could collect telemetry like this for equipment monitoring, environmental sensing, vehicle and robotics diagnostics, or feeding features to an AI model. The main idea: telemetry is structured measurement over time, and a reliable collector with a consistent reading format is what makes everything downstream possible.

---
## Key Terms

- **Telemetry:** measurements collected automatically over time, each tagged with when and where it was taken.
- **Device sensor vs. environment sensor:** a device sensor measures the edge node itself (CPU, memory, temperature); an environment sensor measures the world around it (temperature, motion, a camera).
- **JSON Lines (`.jsonl`):** a file format with one complete JSON object per line — easy to append to and to process one record at a time.
- **Data contract / schema:** a fixed, agreed-upon shape for each reading so anything downstream can parse it without surprises.
- **Sampling rate:** how often you take a reading; higher rates mean more detail but more CPU, storage, and bandwidth.
- **Docker volume:** persistent storage attached to a container so collected data survives the container being stopped or removed.

---
## Cleanup

At the end of the lab, remove only your own container, volume, image, and files.

**[Notebook cell]** Stop and remove the collector container, then the volume and image (safe to run any number of times):

In [ ]:
!docker rm -f ${USER}-telemetryCollector 2>/dev/null || true
!docker volume rm ${USER}-telemetryData 2>/dev/null || true
!docker rmi ${USER}-telemetry-collector 2>/dev/null || true

**[Notebook cell]** Optional — remove your lab folder (uncomment to run). Keep it if the next lab will reuse your telemetry file:

In [ ]:
# !rm -rf ~/telemetryLab

Only remove your own files, containers, images, and volumes.

### Lab scorecard

In [ ]:
labSummary("Device & Sensor Telemetry Collection")

---
### One-minute feedback

Your feedback shapes the next version of this lab. Rate it, add anything that was confusing or broken, and click **Submit**. It takes about 30 seconds and goes straight to the instructor.

In [ ]:
feedback("Device and Sensor Telemetry Collection")